In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_excel(r"C:\Users\LENOVO\Downloads\Unclean_Logistics_Dataset_12000plus_Rows.xlsx")
    
# --------------------------------------------------
# 1. Define column groups
# --------------------------------------------------

categorical_columns = [
    "Origin",
    "Destination",
    "Transport_Mode",
    "Carrier",
    "Shipment_Status",
    "Warehouse_ID",
    "Product_Category"
]

date_columns = [
    "Order_Date",
    "Ship_Date",
    "Delivery_Date"
]

numeric_columns = [
    "Distance_km",
    "Shipment_Weight_kg",
    "Quantity",
    "Shipping_Cost_INR",
    "Promised_Delivery_Days",
    "Actual_Delivery_Days",
    "Customer_Rating",
    "Fuel_Surcharge_INR",
    "Warehouse_Processing_Hours",
    "Temperature_C"
]

# --------------------------------------------------
# 2. Replace blank cells with missing values
# --------------------------------------------------

df = df.replace(r"^\s*$", pd.NA, regex=True)

# --------------------------------------------------
# 3. Remove duplicate rows
# --------------------------------------------------

print("Duplicates before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates after:", df.duplicated().sum())

# --------------------------------------------------
# 4. Clean categorical columns
# --------------------------------------------------

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

# --------------------------------------------------
# 5. Convert date columns
# --------------------------------------------------

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce",
        dayfirst=True
    )

# --------------------------------------------------
# 6. Convert numerical columns
# --------------------------------------------------

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# --------------------------------------------------
# 7. Check missing values
# --------------------------------------------------

print("\nMissing values:")
print(df.isnull().sum())

# --------------------------------------------------
# 8. Handle invalid numerical values
# --------------------------------------------------

# Distance cannot be zero/negative
df.loc[df["Distance_km"] <= 0, "Distance_km"] = np.nan

# Weight cannot be zero/negative
df.loc[df["Shipment_Weight_kg"] <= 0, "Shipment_Weight_kg"] = np.nan

# Shipping cost cannot be zero/negative
df.loc[df["Shipping_Cost_INR"] <= 0, "Shipping_Cost_INR"] = np.nan

# Customer rating should be between 1 and 5
df.loc[
    (df["Customer_Rating"] < 1) |
    (df["Customer_Rating"] > 5),
    "Customer_Rating"
] = np.nan

# --------------------------------------------------
# 9. Fill missing numerical values with median
# --------------------------------------------------

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

# --------------------------------------------------
# 10. Fill missing categorical values with mode
# --------------------------------------------------

for col in categorical_columns:
    if df[col].isnull().any():
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)

# --------------------------------------------------
# 11. Fill missing dates
# --------------------------------------------------

for col in date_columns:
    df[col] = df[col].ffill().bfill()

# --------------------------------------------------
# 12. Create useful logistics features
# --------------------------------------------------

df["Delivery_Delay_Days"] = (
    df["Actual_Delivery_Days"]
    - df["Promised_Delivery_Days"]
)

df["Late_Delivery"] = (
    df["Delivery_Delay_Days"] > 0
).astype(int)

df["Cost_per_km"] = (
    df["Shipping_Cost_INR"] /
    df["Distance_km"]
)

df["Order_Week"] = (
    df["Order_Date"]
    .dt.isocalendar()
    .week
)

df["Order_Month"] = (
    df["Order_Date"]
    .dt.to_period("M")
    .astype(str)
)

# --------------------------------------------------
# 13. Final verification
# --------------------------------------------------

print("\nFinal dataset shape:")
print(df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum().sum())

print("\nFirst 5 rows:")
display(df.head())

Duplicates before: 150
Duplicates after: 0

Missing values:
Shipment_ID                      0
Order_Date                    7377
Ship_Date                     7294
Delivery_Date                 7335
Origin                          20
Destination                      0
Transport_Mode                   0
Carrier                        320
Shipment_Status                  0
Warehouse_ID                     0
Product_Category               140
Distance_km                    177
Shipment_Weight_kg               0
Quantity                         0
Shipping_Cost_INR                0
Promised_Delivery_Days           0
Actual_Delivery_Days             0
Customer_Rating               4088
Fuel_Surcharge_INR               0
Warehouse_Processing_Hours     240
Temperature_C                  180
dtype: int64

Final dataset shape:
(12000, 26)

Remaining missing values:
0

First 5 rows:


,Shipment_ID,Order_Date,Ship_Date,Delivery_Date,Origin,Destination,Transport_Mode,Carrier,Shipment_Status,Warehouse_ID,...,Actual_Delivery_Days,Customer_Rating,Fuel_Surcharge_INR,Warehouse_Processing_Hours,Temperature_C,Delivery_Delay_Days,Late_Delivery,Cost_per_km,Order_Week,Order_Month
0,SHP100000,2025-02-03,2025-04-03,2025-11-03,Jaipur,Kolkata,Air,Bluedart,Delivered,Wh-Che-01,...,7,2.0,4815.69,12.0,21.4,2,1,9.073353,6,2025-02
1,SHP100001,2025-02-03,2025-04-03,2025-11-03,Pune,Pune,Road,Delhivery,Delivered,Wh-Pun-01,...,7,3.0,1018.77,8.6,33.4,0,0,13.400564,6,2025-02
2,SHP100002,2025-02-03,2025-04-03,2025-11-03,Jaipur,Ahmedabad,Road,Ecom Express,Cancelled,Wh-Del-01,...,2,5.0,1107.82,15.6,24.4,-1,0,10.893125,6,2025-02
3,SHP100003,2025-02-03,2025-04-03,2025-11-03,Bengaluru,Lucknow,Road,Delhivery,Delivered,Wh-Che-01,...,3,4.0,1703.07,17.3,23.4,1,1,8.656890,6,2025-02
4,SHP100004,2025-09-12,2025-09-12,2025-11-03,Delhi,Delhi,Air,Dhl,Delivered,Wh-Mum-01,...,9,4.0,96.59,6.1,29.6,2,1,14.083684,37,2025-09
